In [ ]:
%run canvas/OEA_py_canvas

In [ ]:
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F
from datetime import date
import logging
logger = logging.getLogger("CanvasStage1FinalPipeline")

In [ ]:
########################################### update latest with question_no #####################
########### Latest ###########

from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window

def preprocess_canvas_dataset(raw_base_path, version="0.1"):
    """
    Canvas Stage1 FINAL Pipeline
    INPUT:
      stage1/Transactional/canvas_raw/v{version}/{Entity}/{BatchType}/rundate=YYYY-MM-DD

    OUTPUT:
      stage1/Transactional/canvas/v{version}/{student|question}/{BatchType}/rundate=YYYY-MM-DD
    """

    logger.info("Starting Canvas Stage1 FINAL pipeline")
    raw_base_path = raw_base_path.rstrip("/")

    # =====================================================
    # BASE TABLES
    # =====================================================
    base_tables = [
        "Users", "Accounts", "Courses", "Sections",
        "Enrollments", "Quizzes", "Quiz_Questions",
        "Quiz_Submissions", "Outcomes"
    ]

    base_dfs = {}
    batch_type = None
    latest_dt = None

    for table in base_tables:
        table_root = f"{raw_base_path}/v{version}/{table}"

        this_batch_type = oea.get_folders(table_root)[0]
        this_latest_dt = oea.get_latest_runtime(
            f"{table_root}/{this_batch_type}",
            "rundate=%Y-%m-%d"
        ).strftime("%Y-%m-%d")

        source_path = f"{table_root}/{this_batch_type}/rundate={this_latest_dt}"

        df = oea.load_json(source_path, multiline=True)
        base_dfs[table] = df

        if batch_type is None:
            batch_type = this_batch_type
            latest_dt = this_latest_dt

        logger.info(f"Loaded base table {table}")

    # =====================================================
    # UNPACK
    # =====================================================
    users = base_dfs["Users"]
    accounts = base_dfs["Accounts"]
    courses = base_dfs["Courses"]
    sections = base_dfs["Sections"]
    enrollments = base_dfs["Enrollments"]
    quizzes = base_dfs["Quizzes"]
    quiz_items = base_dfs["Quiz_Questions"]
    quiz_submissions = base_dfs["Quiz_Submissions"]
    outcomes = base_dfs["Outcomes"]

    # =====================================================
    # SECTION INSTRUCTORS
    # =====================================================
    section_instructors = (
        enrollments
        .filter(F.col("type") == "TeacherEnrollment")
        .join(users, enrollments.user_id == users.id)
        .groupBy(enrollments.section_id)
        .agg(
            F.concat_ws(", ", F.collect_set(users.name))
            .alias("Section_Instructors")
        )
    )

    # =====================================================
    # STUDENT BUILDER
    # =====================================================
    def build_student():
        # -----------------------
        # RENAME / ALIAS SOURCES
        # -----------------------
        users_r = users.withColumnRenamed("id", "user_id_dim").alias("usr")

        enrollments_r = (
            enrollments
            .withColumnRenamed("user_id", "user_id_enr")
            .withColumnRenamed("id", "enrollment_id")
            .alias("enr")
        )

        quiz_sub_r = (
            quiz_submissions
            .withColumnRenamed("user_id", "user_id_sub")
            .withColumnRenamed("quiz_id", "quiz_id_sub")
            .alias("qs")
        )

        quizzes_r = quizzes.withColumnRenamed("id", "quiz_id_dim").alias("qz")
        courses_r = courses.withColumnRenamed("id", "course_id_dim").alias("crs")
        sections_r = sections.withColumnRenamed("id", "section_id_dim").alias("sec")

        # -----------------------
        # ACCOUNT HIERARCHY
        # -----------------------
        grade_accounts = (
            accounts
            .withColumnRenamed("id", "grade_id")
            .withColumnRenamed("name", "Grade_Name")
            .withColumnRenamed("parent_account_id", "school_id")
            .alias("grade_acc")
        )

        school_accounts = (
            accounts
            .withColumnRenamed("id", "school_id")
            .withColumnRenamed("name", "User_School_Name")
            .withColumnRenamed("parent_account_id", "district_id")
            .alias("school_acc")
        )

        # -----------------------
        # LATEST ATTEMPT (QUESTION LEVEL TIMESTAMP)
        # -----------------------
        la_q = (
            quiz_sub_r
            .withColumn("question", F.explode("questions"))
            .withColumn("question_id", F.col("question.item_id"))
            .withColumn(
                "attempt_ts",
                F.coalesce(
                    F.col("submitted_at"),
                    F.col("finished_at"),
                    F.col("started_at")
                )
            )
        )

        latest_attempts = (
            la_q
            .groupBy("user_id_sub", "quiz_id_sub", "question_id")
            .agg(F.max("attempt_ts").alias("Latest_Attempt"))
            .alias("la")
        )

        # -----------------------
        # EXPLODE QUESTIONS
        # -----------------------
        qs = quiz_sub_r.withColumn("question", F.explode("questions"))

        # -----------------------
        # SUB-QUESTION LOGIC
        # -----------------------
        base = (
            qs
            .withColumn("question_id", F.col("question.item_id"))
            .withColumn("raw_answer", F.col("question.answer").cast("string"))
        )

        base = base.withColumn(
            "answer_map",
            F.from_json("raw_answer", MapType(StringType(), StringType()))
        )

        base = base.withColumn("is_multipart", F.col("answer_map").isNotNull())

        multipart = (
            base.filter("is_multipart")
            .select("*", F.explode("answer_map").alias("part_key", "part_answer"))
            .withColumn(
                "Sub_Question",
                F.concat_ws("::", F.col("question_id"), F.col("part_key"))
            )
            .withColumn("Answer_Submission", F.col("part_answer"))
        )

        singlepart = (
            base.filter("NOT is_multipart")
            .withColumn("Sub_Question", F.lit(None).cast("string"))
            .withColumn("Answer_Submission", F.col("raw_answer"))
        )

        norm = multipart.select(singlepart.columns).unionByName(singlepart).alias("n")

        # -----------------------
        # FINAL JOIN GRAPH
        # -----------------------
        final_df = (
            norm
            .join(users_r, F.col("n.user_id_sub") == F.col("usr.user_id_dim"))
            .join(enrollments_r, F.col("n.user_id_sub") == F.col("enr.user_id_enr"))
            .join(sections_r, F.col("enr.section_id") == F.col("sec.section_id_dim"))
            .join(courses_r, F.col("sec.course_id") == F.col("crs.course_id_dim"))

            .join(grade_accounts, F.col("crs.account_id") == F.col("grade_acc.grade_id"), "left")
            .join(school_accounts, F.col("grade_acc.school_id") == F.col("school_acc.school_id"), "left")

            .join(quizzes_r, F.col("n.quiz_id_sub") == F.col("qz.quiz_id_dim"))
            .join(quiz_items.alias("qi"), F.col("n.question_id") == F.col("qi.id"))

            .join(
                section_instructors.alias("si"),
                F.col("sec.section_id_dim") == F.col("si.section_id"),
                "left"
            )

            # QUESTION LEVEL LATEST ATTEMPT JOIN
            .join(
                latest_attempts,
                (F.col("n.user_id_sub") == F.col("la.user_id_sub")) &
                (F.col("n.quiz_id_sub") == F.col("la.quiz_id_sub")) &
                (F.col("n.question_id") == F.col("la.question_id")),
                "left"
            )
        )

        # -----------------------
        # ACADEMIC SESSION FORMAT (2025-26)
        # -----------------------
        session_col = F.concat(
            F.when(F.month("qz.due_at") >= 7, F.year("qz.due_at"))
            .otherwise(F.year("qz.due_at") - 1)
            .cast("string"),

            F.lit("-"),

            F.substring(
                (
                    F.when(F.month("qz.due_at") >= 7, F.year("qz.due_at") + 1)
                    .otherwise(F.year("qz.due_at"))
                ).cast("string"),
                -2,  # take last 2 characters
                2
            )
        )

        # -----------------------
        # SELECT
        # -----------------------
        return final_df.select(
            F.col("usr.user_id_dim").cast("string").alias("User_UID"),
            F.col("usr.login_id").cast("string").alias("Username"),
            F.split(F.col("usr.name"), " ").getItem(1).cast("string").alias("Last_Name"),
            F.split(F.col("usr.name"), " ").getItem(0).cast("string").alias("First_Name"),

            F.col("enr.type").cast("string").alias("User_Role_ID"),
            F.col("enr.role").cast("string").alias("User_role_name"),

            F.col("school_acc.school_id").cast("string").alias("User_School_ID"),
            F.col("school_acc.User_School_Name").cast("string").alias("User_School_Name"),

            F.col("crs.course_id_dim").cast("string").alias("Course_NID"),
            F.col("crs.name").cast("string").alias("Course_Name"),
            F.col("crs.course_code").cast("string").alias("Course_code"),

            F.col("sec.section_id_dim").cast("string").alias("Section_NID"),
            F.col("sec.name").cast("string").alias("Section_Name"),
            F.col("sec.sis_section_id").cast("string").alias("Section_Code"),

            F.col("qz.quiz_type").cast("string").alias("Item_Type"),
            F.col("qz.quiz_id_dim").cast("string").alias("Item_ID"),
            F.col("qz.title").cast("string").alias("Item_Name"),

            F.col("n.started_at").cast("string").alias("First_Access"),
            F.col("la.Latest_Attempt").cast("string").alias("Latest_Attempt"),
            F.col("n.time_spent").cast("string").alias("Total_Time"),
            F.col("n.score").cast("string").alias("Submission_Grade"),
            F.col("n.attempt").cast("string").alias("Submission"),

            F.col("qi.id").cast("string").alias("Question_ID"),
            F.lit(None).cast("string").alias("Associated_Question_ID"),
            F.col("qi.interaction_type").cast("string").alias("Question_Type"),
            F.col("qi.interaction_data.prompt").cast("string").alias("Question"),
            F.col("qi.position").cast("string").alias("Position_Number"),

            F.col("n.Sub_Question").cast("string").alias("Sub-Question"),
            F.col("n.Answer_Submission").cast("string").alias("Answer_Submission"),
            F.concat_ws("|", F.col("qi.interaction_data.correct_responses"))
                .cast("string").alias("Correct_Answer"),

            F.col("question.score").cast("string").alias("Points_Received"),
            F.col("qi.scoring_data.points_possible").cast("string").alias("Points_Possible"),

            F.sha2(
                F.concat_ws(
                    "",
                    F.col("usr.user_id_dim"),
                    F.col("qz.quiz_id_dim"),
                    F.col("qi.id"),
                    F.col("qi.position"),
                    F.col("n.Answer_Submission"),
                    F.col("question.score"),
                    F.col("qi.scoring_data.points_possible"),
                    F.col("n.attempt"),
                    F.concat_ws("|", F.col("qi.interaction_data.correct_responses"))
                ),
                256
            ).alias("Unique_Key"),

            session_col.cast("string").alias("Session"),
            F.col("qz.quiz_type").cast("string").alias("Assessment_type"),
            F.col("crs.name").cast("string").alias("Subject"),
            F.col("grade_acc.Grade_Name").cast("string").alias("Grade"),
            F.col("sec.name").cast("string").alias("Section"),
            F.concat_ws("_", F.col("qz.title"), F.col("qz.due_at"))
                .cast("string").alias("File_Name"),

            F.col("si.Section_Instructors").cast("string").alias("Section_Instructors")
        )

    # def build_student():

    #     # -----------------------
    #     # RENAME / ALIAS SOURCES
    #     # -----------------------
    #     users_r = users.withColumnRenamed("id", "user_id_dim").alias("usr")

    #     enrollments_r = (
    #         enrollments
    #         .withColumnRenamed("user_id", "user_id_enr")
    #         .withColumnRenamed("id", "enrollment_id")
    #         .alias("enr")
    #     )

    #     quiz_sub_r = (
    #         quiz_submissions
    #         .withColumnRenamed("user_id", "user_id_sub")
    #         .withColumnRenamed("quiz_id", "quiz_id_sub")
    #         .alias("qs")
    #     )

    #     quizzes_r = quizzes.withColumnRenamed("id", "quiz_id_dim").alias("qz")
    #     courses_r = courses.withColumnRenamed("id", "course_id_dim").alias("crs")
    #     sections_r = sections.withColumnRenamed("id", "section_id_dim").alias("sec")

    #     # -----------------------
    #     # ACCOUNT HIERARCHY
    #     # -----------------------
    #     grade_accounts = (
    #         accounts
    #         .withColumnRenamed("id", "grade_id")
    #         .withColumnRenamed("name", "Grade_Name")
    #         .withColumnRenamed("parent_account_id", "school_id")
    #         .alias("grade_acc")
    #     )

    #     school_accounts = (
    #         accounts
    #         .withColumnRenamed("id", "school_id")
    #         .withColumnRenamed("name", "User_School_Name")
    #         .withColumnRenamed("parent_account_id", "district_id")
    #         .alias("school_acc")
    #     )

    #     # -----------------------
    #     # LATEST ATTEMPT
    #     # -----------------------
    #     latest_attempts = (
    #         quiz_sub_r
    #         .groupBy("user_id_sub", "quiz_id_sub")
    #         .agg(F.max("attempt").alias("Latest_Attempt"))
    #         .alias("la")
    #     )

    #     # -----------------------
    #     # EXPLODE QUESTIONS
    #     # -----------------------
    #     qs = quiz_sub_r.withColumn("question", F.explode("questions"))

    #     # -----------------------
    #     # SUB-QUESTION LOGIC
    #     # -----------------------
    #     base = (
    #         qs
    #         .withColumn("question_id", F.col("question.item_id"))
    #         .withColumn("raw_answer", F.col("question.answer").cast("string"))
    #     )

    #     base = base.withColumn(
    #         "answer_map",
    #         F.from_json("raw_answer", MapType(StringType(), StringType()))
    #     )

    #     base = base.withColumn("is_multipart", F.col("answer_map").isNotNull())

    #     multipart = (
    #         base.filter("is_multipart")
    #         .select("*", F.explode("answer_map").alias("part_key", "part_answer"))
    #         .withColumn(
    #             "Sub_Question",
    #             F.concat_ws("::", F.col("question_id"), F.col("part_key"))
    #         )
    #         .withColumn("Answer_Submission", F.col("part_answer"))
    #     )

    #     singlepart = (
    #         base.filter("NOT is_multipart")
    #         .withColumn("Sub_Question", F.lit(None).cast("string"))
    #         .withColumn("Answer_Submission", F.col("raw_answer"))
    #     )

    #     norm = multipart.select(singlepart.columns).unionByName(singlepart).alias("n")

    #     # -----------------------
    #     # FINAL JOIN GRAPH
    #     # -----------------------
    #     final_df = (
    #         norm
    #         .join(users_r, F.col("n.user_id_sub") == F.col("usr.user_id_dim"))
    #         .join(enrollments_r, F.col("n.user_id_sub") == F.col("enr.user_id_enr"))
    #         .join(sections_r, F.col("enr.section_id") == F.col("sec.section_id_dim"))
    #         .join(courses_r, F.col("sec.course_id") == F.col("crs.course_id_dim"))

    #         # Grade → School
    #         .join(grade_accounts, F.col("crs.account_id") == F.col("grade_acc.grade_id"), "left")
    #         .join(school_accounts, F.col("grade_acc.school_id") == F.col("school_acc.school_id"), "left")

    #         # Quiz / Items
    #         .join(quizzes_r, F.col("n.quiz_id_sub") == F.col("qz.quiz_id_dim"))
    #         .join(quiz_items.alias("qi"), F.col("n.question_id") == F.col("qi.id"))

    #         # Instructors
    #         .join(
    #             section_instructors.alias("si"),
    #             F.col("sec.section_id_dim") == F.col("si.section_id"),
    #             "left"
    #         )

    #         # Latest Attempt
    #         .join(
    #             latest_attempts,
    #             (F.col("n.user_id_sub") == F.col("la.user_id_sub")) &
    #             (F.col("n.quiz_id_sub") == F.col("la.quiz_id_sub")),
    #             "left"
    #         )
    #     )

    #     # -----------------------
    #     # SELECT (ONLY REQUIRED COLUMNS — ALL STRING, ALIAS-SAFE)
    #     # -----------------------
    #     return final_df.select(
    #         # F.lit(None).cast("string").alias("_c0"),

    #         F.col("usr.user_id_dim").cast("string").alias("User_UID"),
    #         F.col("usr.login_id").cast("string").alias("Username"),
    #         F.split(F.col("usr.name"), " ").getItem(1).cast("string").alias("Last_Name"),
    #         F.split(F.col("usr.name"), " ").getItem(0).cast("string").alias("First_Name"),

    #         F.col("enr.type").cast("string").alias("User_Role_ID"),
    #         F.col("enr.role").cast("string").alias("User_role_name"),

    #         F.col("school_acc.school_id").cast("string").alias("User_School_ID"),
    #         F.col("school_acc.User_School_Name").cast("string").alias("User_School_Name"),

    #         F.col("crs.course_id_dim").cast("string").alias("Course_NID"),
    #         F.col("crs.name").cast("string").alias("Course_Name"),
    #         F.col("crs.course_code").cast("string").alias("Course_code"),

    #         F.col("sec.section_id_dim").cast("string").alias("Section_NID"),
    #         F.col("sec.name").cast("string").alias("Section_Name"),
    #         F.col("sec.sis_section_id").cast("string").alias("Section_Code"),

    #         F.col("qz.quiz_type").cast("string").alias("Item_Type"),
    #         F.col("qz.quiz_id_dim").cast("string").alias("Item_ID"),
    #         F.col("qz.title").cast("string").alias("Item_Name"),

    #         # THESE COME FROM norm (alias n), NOT qs
    #         F.col("n.started_at").cast("string").alias("First_Access"),
    #         F.col("la.Latest_Attempt").cast("string").alias("Latest_Attempt"),
    #         F.col("n.time_spent").cast("string").alias("Total_Time"),
    #         F.col("n.score").cast("string").alias("Submission_Grade"),
    #         F.col("n.attempt").cast("string").alias("Submission"),

    #         F.col("qi.id").cast("string").alias("Question_ID"),
    #         F.lit(None).cast("string").alias("Associated_Question_ID"),
    #         F.col("qi.interaction_type").cast("string").alias("Question_Type"),
    #         F.col("qi.interaction_data.prompt").cast("string").alias("Question"),
    #         F.col("qi.position").cast("string").alias("Position_Number"),

    #         F.col("n.Sub_Question").cast("string").alias("Sub-Question"),
    #         F.col("n.Answer_Submission").cast("string").alias("Answer_Submission"),
    #         F.concat_ws("|", F.col("qi.interaction_data.correct_responses"))
    #             .cast("string").alias("Correct_Answer"),

    #         F.col("question.score").cast("string").alias("Points_Received"),
    #         F.col("qi.scoring_data.points_possible").cast("string").alias("Points_Possible"),

    #         # UNIQUE KEY
    #         # F.concat_ws(
    #         #     "",
    #         #     F.coalesce(F.col("usr.user_id_dim").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("qz.quiz_id_dim").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("qi.id").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("qi.position").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("n.Answer_Submission").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("question.score").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("qi.scoring_data.points_possible").cast("string"), F.lit("")),
    #         #     F.coalesce(F.col("n.attempt").cast("string"), F.lit("")),
    #         #     F.coalesce(
    #         #         F.concat_ws("|", F.col("qi.interaction_data.correct_responses")),
    #         #         F.lit("")
    #         #     )
    #         # ).alias("Unique_Key"),
    #         F.sha2(
    #             F.concat_ws(
    #                 "",
    #                 F.coalesce(F.col("usr.user_id_dim").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("qz.quiz_id_dim").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("qi.id").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("qi.position").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("n.Answer_Submission").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("question.score").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("qi.scoring_data.points_possible").cast("string"), F.lit("")),
    #                 F.coalesce(F.col("n.attempt").cast("string"), F.lit("")),
    #                 F.coalesce(
    #                     F.concat_ws("|", F.col("qi.interaction_data.correct_responses")),
    #                     F.lit("")
    #                 )
    #             ),
    #             256
    #         ).alias("Unique_Key"),

    #         F.col("qz.due_at").cast("string").alias("Session"),
    #         F.col("qz.quiz_type").cast("string").alias("Assessment_type"),
    #         F.col("crs.name").cast("string").alias("Subject"),
    #         F.col("grade_acc.Grade_Name").cast("string").alias("Grade"),
    #         F.col("sec.name").cast("string").alias("Section"),
    #         F.concat_ws("_", F.col("qz.title"), F.col("qz.due_at"))
    #             .cast("string").alias("File_Name"),

    #         F.col("si.Section_Instructors").cast("string").alias("Section_Instructors")
    #     )

    # =====================================================
    # QUESTION BUILDER
    # =====================================================
    def build_question():

        qs_src = quiz_submissions.alias("qs")
        qi_src = quiz_items.alias("qi")
        qz_src = quizzes.alias("qz")
        crs_src = courses.alias("crs")
        acc_g = accounts.alias("grade_acc")
        acc_s = accounts.alias("school_acc")
        sec_src = sections.alias("sec")
        out_src = outcomes.alias("out")

        qs = qs_src.withColumn("question", F.explode(F.col("qs.questions")))

        base = (
            qs
            .withColumn("question_id", F.col("question.item_id"))
            .withColumn("raw_answer", F.col("question.answer").cast("string"))
        )

        base = base.withColumn(
            "answer_map",
            F.from_json("raw_answer", MapType(StringType(), StringType()))
        )

        base = base.withColumn("is_multipart", F.col("answer_map").isNotNull())

        multipart = (
            base.filter("is_multipart")
            .select("*", F.explode("answer_map").alias("part_key", "part_answer"))
            .withColumn(
                "Sub_Question",
                F.concat_ws("::", F.col("question_id"), F.col("part_key"))
            )
            .withColumn("answer_id", F.col("part_answer"))
        )

        singlepart = (
            base.filter("NOT is_multipart")
            .withColumn("Sub_Question", F.lit(None).cast("string"))
            .withColumn("answer_id", F.col("raw_answer"))
        )

        norm = multipart.select(singlepart.columns).unionByName(singlepart)

        stats = (
            norm.groupBy("question_id", "Sub_Question")
            .agg(
                F.min("question.score").alias("Least_Points_Earned"),
                F.avg("question.score").alias("Average_Points_Earned"),
                F.max("question.score").alias("Most_Points_Earned"),
                F.count("*").alias("Total_Attempts"),
                F.sum(
                    F.when(F.col("question.score") > 0, 1).otherwise(0)
                ).alias("Correct_Count")
            )
            .alias("st")
        )

        answer_counts = (
            norm.groupBy(
                F.col("question_id").alias("item_id"),
                "Sub_Question",
                "answer_id"
            )
            .agg(F.count("*").alias("answer_count"))
            .alias("ac")
        )

        qi = qi_src.withColumn(
            "choice",
            F.explode(F.col("qi.interaction_data.choices"))
        ).alias("qi")

        breakdown = (
            qi
            .join(answer_counts, F.col("qi.id") == F.col("ac.item_id"), "left")
            .withColumn("answer_count", F.coalesce(F.col("ac.answer_count"), F.lit(0)))
            .groupBy(F.col("qi.id"), F.col("ac.Sub_Question"))
            .agg(
                F.concat_ws(
                    " | ",
                    F.collect_list(
                        F.concat(
                            F.col("choice.text"),
                            F.lit(":"),
                            F.col("answer_count").cast("string")
                        )
                    )
                ).alias("Answer_Breakdown"),
                F.concat_ws(
                    ",",
                    F.collect_list(
                        F.concat(
                            F.col("choice.id"),
                            F.lit("="),
                            F.col("answer_count").cast("string")
                        )
                    )
                ).alias("Answer_Breakdown_Val")
            )
            .alias("bd")
        )

        final_df = (
            qi
            .join(qz_src, F.col("qi.quiz_id") == F.col("qz.id"))
            .join(crs_src, F.col("qz.course_id") == F.col("crs.id"))

            .join(acc_g, F.col("crs.account_id") == F.col("grade_acc.id"), "left")
            .join(acc_s, F.col("grade_acc.parent_account_id") == F.col("school_acc.id"), "left")

            .join(sec_src, F.col("crs.id") == F.col("sec.course_id"), "left")

            .join(stats, F.col("qi.id") == F.col("st.question_id"), "left")
            .join(out_src, F.col("qi.learning_outcome_id") == F.col("out.id"), "left")
            .join(
                breakdown,
                (F.col("qi.id") == F.col("bd.id")) &
                (F.col("st.Sub_Question") == F.col("bd.Sub_Question")),
                "left"
            )
        )

        # DENSE RANK (PANDAS rankdata equivalent)
        w = Window.orderBy(F.col("qi.id"))
        final_df = final_df.withColumn("Question_No", F.dense_rank().over(w))

        # -----------------------
        # ACADEMIC SESSION FORMAT (2025-26)
        # -----------------------
        session_col = F.concat(
            F.when(F.month("qz.due_at") >= 7, F.year("qz.due_at"))
            .otherwise(F.year("qz.due_at") - 1)
            .cast("string"),

            F.lit("-"),

            F.substring(
                (
                    F.when(F.month("qz.due_at") >= 7, F.year("qz.due_at") + 1)
                    .otherwise(F.year("qz.due_at"))
                ).cast("string"),
                -2,  # take last 2 characters
                2
            )
        )
        return final_df.select(
            F.col("qi.interaction_data.prompt").cast("string").alias("Question"),
            F.col("out.id").cast("string").alias("Standards"), # Standards_Val to Standards
            F.col("out.title").cast("string").alias("Standards_Val"), # Standards to Standards_Val 
            F.col("qi.position").cast("string").alias("Position_Number"),
            F.col("qz.id").cast("string").alias("Item_ID"),
            F.col("qi.id").cast("string").alias("Question_ID"),

            F.col("st.Least_Points_Earned").cast("string").alias("Least_Points_Earned"),
            F.concat_ws("|", F.col("qi.interaction_data.correct_responses"))
                .cast("string").alias("Correct_Answer"),
            F.col("qi.interaction_type").cast("string").alias("Question_Type"),
            F.col("choice.text").cast("string").alias("Answer_Option"),
            F.col("qz.title").cast("string").alias("Item_Name"),
            F.col("st.Average_Points_Earned").cast("string").alias("Average_Points_Earned"),

            F.lit(None).cast("string").alias("Associated_Question_ID"),
            F.col("qi.scoring_data.points_possible").cast("string").alias("Total_Points"),
            F.col("st.Most_Points_Earned").cast("string").alias("Most_Points_Earned"),

            F.round(
                F.when(
                    F.col("qi.scoring_data.points_possible") > 0,
                    F.col("st.Average_Points_Earned") /
                    F.col("qi.scoring_data.points_possible")
                ).otherwise(F.lit(0)),
                2
            ).cast("string").alias("Correctly_Answered"),

            # ✅ Academic Session (2025-26 format)
            session_col.cast("string").alias("Session"),

            F.col("st.Sub_Question").cast("string").alias("Sub-Question"),
            F.col("bd.Answer_Breakdown").cast("string").alias("Answer_Breakdown"),
            F.col("bd.Answer_Breakdown_Val").cast("string").alias("Answer_Breakdown_Val"),

            F.sha2(
                F.concat_ws(
                    "",
                    F.coalesce(F.col("qz.id").cast("string"), F.lit("")),
                    F.coalesce(F.col("qi.id").cast("string"), F.lit("")),
                    F.coalesce(
                        F.concat_ws("|", F.col("qi.interaction_data.correct_responses")),
                        F.lit("")
                    ),
                    F.coalesce(F.col("qi.position").cast("string"), F.lit("")),
                    F.coalesce(F.col("choice.text").cast("string"), F.lit("")),
                    F.coalesce(F.col("bd.Answer_Breakdown").cast("string"), F.lit("")),
                    F.coalesce(F.col("out.title").cast("string"), F.lit(""))
                ),
                256
            ).alias("Unique_Key"),

            F.col("Question_No").cast("string").alias("Question_No"),

            # # ✅ Renamed to avoid collision
            # F.col("qz.due_at").cast("string").alias("Quiz_Due_At"),

            F.col("qz.quiz_type").cast("string").alias("Assessment_type"),
            F.col("crs.name").cast("string").alias("Subject"),
            F.col("grade_acc.name").cast("string").alias("Grade"),
            F.col("sec.name").cast("string").alias("Section"),

            F.concat_ws("_", F.col("qz.title"), F.col("qz.due_at"))
                .cast("string").alias("File_Name")
        )
        # return final_df.select(
        #     # F.lit(None).cast("string").alias("_c0"),

        #     F.col("qi.interaction_data.prompt").cast("string").alias("Question"),
        #     F.col("out.id").cast("string").alias("Standards_Val"),
        #     F.col("out.title").cast("string").alias("Standards"),
        #     F.col("qi.position").cast("string").alias("Position_Number"),
        #     F.col("qz.id").cast("string").alias("Item_ID"),
        #     F.col("qi.id").cast("string").alias("Question_ID"),

        #     F.col("st.Least_Points_Earned").cast("string").alias("Least_Points_Earned"),
        #     F.concat_ws("|", F.col("qi.interaction_data.correct_responses"))
        #         .cast("string").alias("Correct_Answer"),
        #     F.col("qi.interaction_type").cast("string").alias("Question_Type"),
        #     F.col("choice.text").cast("string").alias("Answer_Option"),
        #     F.col("qz.title").cast("string").alias("Item_Name"),
        #     F.col("st.Average_Points_Earned").cast("string").alias("Average_Points_Earned"),

        #     F.lit(None).cast("string").alias("Associated_Question_ID"),
        #     F.col("qi.scoring_data.points_possible").cast("string").alias("Total_Points"),
        #     F.col("st.Most_Points_Earned").cast("string").alias("Most_Points_Earned"),

        #     F.round(
        #         F.when(
        #             F.col("qi.scoring_data.points_possible") > 0,
        #             F.col("st.Average_Points_Earned") /
        #             F.col("qi.scoring_data.points_possible")
        #         ).otherwise(F.lit(0)),
        #         2
        #     ).cast("string").alias("Correctly_Answered"),
        #     session_col.cast("string").alias("Session"),
        #     F.col("st.Sub_Question").cast("string").alias("Sub-Question"),
        #     F.col("bd.Answer_Breakdown").cast("string").alias("Answer_Breakdown"),
        #     F.col("bd.Answer_Breakdown_Val").cast("string").alias("Answer_Breakdown_Val"),

        #     # UNIQUE KEY
        #     # F.concat_ws(
        #     #     "",
        #     #     F.coalesce(F.col("qz.id").cast("string"), F.lit("")),
        #     #     F.coalesce(F.col("qi.id").cast("string"), F.lit("")),
        #     #     F.coalesce(
        #     #         F.concat_ws("|", F.col("qi.interaction_data.correct_responses")),
        #     #         F.lit("")
        #     #     ),
        #     #     F.coalesce(F.col("qi.position").cast("string"), F.lit("")),
        #     #     F.coalesce(F.col("choice.text").cast("string"), F.lit("")),
        #     #     F.coalesce(F.col("bd.Answer_Breakdown").cast("string"), F.lit("")),
        #     #     F.coalesce(F.col("out.title").cast("string"), F.lit(""))
        #     # ).alias("Unique_Key"),
        #     F.sha2(
        #         F.concat_ws(
        #             "",
        #             F.coalesce(F.col("qz.id").cast("string"), F.lit("")),
        #             F.coalesce(F.col("qi.id").cast("string"), F.lit("")),
        #             F.coalesce(
        #                 F.concat_ws("|", F.col("qi.interaction_data.correct_responses")),
        #                 F.lit("")
        #             ),
        #             F.coalesce(F.col("qi.position").cast("string"), F.lit("")),
        #             F.coalesce(F.col("choice.text").cast("string"), F.lit("")),
        #             F.coalesce(F.col("bd.Answer_Breakdown").cast("string"), F.lit("")),
        #             F.coalesce(F.col("out.title").cast("string"), F.lit(""))
        #         ),
        #         256
        #     ).alias("Unique_Key"),

        #     F.col("Question_No").cast("string").alias("Question_No"),
        #     F.col("qz.due_at").cast("string").alias("Session"),
        #     F.col("qz.quiz_type").cast("string").alias("Assessment_type"),
        #     F.col("crs.name").cast("string").alias("Subject"),
        #     F.col("grade_acc.name").cast("string").alias("Grade"),
        #     F.col("sec.name").cast("string").alias("Section"),
        #     F.concat_ws("_", F.col("qz.title"), F.col("qz.due_at"))
        #         .cast("string").alias("File_Name")
        # )

    # =====================================================
    # MAIN LOOP
    # =====================================================
    ENTITY_BUILDERS = {
        "student_submissions": build_student,
        "question_data": build_question
    }

    for output_name, builder in ENTITY_BUILDERS.items():
        logger.info(f"Building dataset: {output_name}")

        try:
            df = builder()
            display(df)
            logger.info(f"Built {output_name} successfully")
        except Exception as e:
            logger.error(f"Failed building {output_name}: {e}", exc_info=True)
            continue

        entity_path = f"canvas/v{version}/{output_name}"

        oea.land(df, entity_path=entity_path, rundate=latest_dt)

        new_table_path = (
            f"stage1/Transactional/canvas/"
            f"v{version}/{output_name}/{batch_type}/rundate={latest_dt}"
        )
        oea.rm_if_exists(new_table_path + "/_SUCCESS", False)

        logger.info(f"Finished dataset: {output_name}")

    logger.info("Canvas Stage1 FINAL pipeline completed successfully")


# ############## standard ingestion in stage1 raw ##################
# source_path = "stage1/Transactional/canvas_raw/v0.1/standards/delta_batch_data/rundate=2024-09-06"
# df = oea.load_json(source_path, multiline=True)
# path = "canvas/v0.1/standards"
# oea.land(df, entity_path=path,filename="standards.json")


In [ ]:
preprocess_canvas_dataset("stage1/Transactional/canvas_raw",0.1)